# Query translation for the translate-then-search control (decision 0019)

IndicTrans2 1B (Indic to English) translates every Hindi and Kannada benchmark query
(`data/queries/translation_input_v1.tsv`, written by `src.queries.translate input`) with the settings of its model
card: beam search with 5 beams, maximum length 256, IndicTransToolkit 1.1.1 for the pre- and post-processing.
Only queries are translated, about 40 short lines, so any free GPU (or even the CPU) is enough.

**Before running**
- The model is gated: on https://huggingface.co/ai4bharat/indictrans2-indic-en-1B, log in and accept the conditions
  (approved automatically). Then save a Hugging Face token as a secret named `HF_TOKEN` (Kaggle: Add-ons, Secrets;
  Colab: the key icon).
- Kaggle or Colab, GPU, internet on.

**Output**: `query_translation_v1/translations_v1.tsv` (query id, language, original, English) and
`translations_v1.json` (model, revision, toolkit version, settings, date). Place both in `data/queries/`, then run
`uv run python -m src.queries.translate check`. Translations are never edited; the author's adequacy marks go in a
separate file.

In [ ]:
REPO = "https://github.com/azzs11/heritagerec.git"
BRANCH = "main"
MODEL = "ai4bharat/indictrans2-indic-en-1B"  # MIT; gated (accept the conditions on the model page)
TAGS = {"hi": "hin_Deva", "kn": "kan_Knda"}
TARGET = "eng_Latn"
GENERATION = {"num_beams": 5, "max_length": 256, "num_return_sequences": 1, "min_length": 0, "use_cache": True}
OUT_DIR = "query_translation_v1"

In [ ]:
!git clone --depth 1 -b $BRANCH $REPO heritagerec
!pip install -q "indictranstoolkit==1.1.1" "transformers>=4.51" sentencepiece

In [ ]:
# The gated model needs a token: Kaggle and Colab keep it in different places.
from huggingface_hub import login

token = None
try:
    from kaggle_secrets import UserSecretsClient

    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata

        token = userdata.get("HF_TOKEN")
    except Exception as e:
        print("No token found:", type(e).__name__)
if token:
    login(token=token)

In [ ]:
import csv
import json
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import torch
import transformers
from huggingface_hub import HfApi
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

try:
    from IndicTransToolkit.processor import IndicProcessor  # the model card's import
except ImportError:
    from IndicTransToolkit import IndicProcessor  # the toolkit README's import

rows = list(csv.reader(open("heritagerec/data/queries/translation_input_v1.tsv", encoding="utf-8"), delimiter="\t"))
device = "cuda" if torch.cuda.is_available() else "cpu"
revision = HfApi().model_info(MODEL).sha  # recorded, and the exact revision loaded
tokenizer = AutoTokenizer.from_pretrained(MODEL, revision=revision, trust_remote_code=True)
dtype = torch.float16 if device == "cuda" else torch.float32
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, revision=revision, trust_remote_code=True, torch_dtype=dtype).to(
    device
)
model.eval()
processor = IndicProcessor(inference=True)
print(len(rows), "queries;", device, revision)

In [ ]:
out = []
for lang, tag in TAGS.items():
    group = [r for r in rows if r[1] == lang]
    if not group:
        continue
    batch = processor.preprocess_batch([r[2] for r in group], src_lang=tag, tgt_lang=TARGET)
    inputs = tokenizer(batch, truncation=True, padding="longest", return_tensors="pt", return_attention_mask=True).to(
        device
    )
    with torch.no_grad():
        generated = model.generate(**inputs, **GENERATION)
    decoded = tokenizer.batch_decode(generated, skip_special_tokens=True, clean_up_tokenization_spaces=True)
    english = processor.postprocess_batch(decoded, lang=TARGET)
    for (qid, _, original), text in zip(group, english):
        out.append([qid, lang, original, " ".join(text.split())])
        print(qid, lang, "|", original, "->", text)

Path(OUT_DIR).mkdir(exist_ok=True)
with open(Path(OUT_DIR) / "translations_v1.tsv", "w", encoding="utf-8", newline="") as f:
    csv.writer(f, delimiter="\t", lineterminator="\n").writerows(out)
meta = {
    "model": MODEL,
    "revision": revision,
    "indictranstoolkit": version("indictranstoolkit"),
    "transformers": transformers.__version__,
    "torch": torch.__version__,
    "device": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
    "dtype": str(dtype).replace("torch.", ""),
    "tags": TAGS,
    "target": TARGET,
    "generation": GENERATION,
    "date": datetime.now(timezone.utc).isoformat(),
}
(Path(OUT_DIR) / "translations_v1.json").write_text(json.dumps(meta, indent=2) + "\n")
print(len(out), "translations")

In [ ]:
try:
    from google.colab import files

    for name in ("translations_v1.tsv", "translations_v1.json"):
        files.download(str(Path(OUT_DIR) / name))
except ImportError:
    # On Kaggle the files are in the Output panel under query_translation_v1/.
    print("Download", Path(OUT_DIR).resolve(), "from the notebook's output files")